In [ ]:
import os
import re
import json
import datetime
import anthropic
import multiprocessing
from functools import partial
from tqdm import tqdm
from dotenv import load_dotenv
from openai import OpenAIError
from langchain_openai import ChatOpenAI
from langchain_anthropic import ChatAnthropic
from langchain_deepseek import ChatDeepSeek
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_mistralai.chat_models import ChatMistralAI
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_xai import ChatXAI
from langchain.schema import HumanMessage, SystemMessage
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type
from datasets import load_dataset
from pathlib import Path
from module.utils import (
    get_tree, 
    get_folder_description, 
    extract_module_description, 
    extract_modified_file_path, 
    generate_tree_string, 
    build_tree_dict, 
    generate_tree_json
)

In [ ]:
def load_environment():
    """Load environment variables"""
    load_dotenv()

def get_system_prompt(config):
    prompts = {
        "chatgpt": f"""
You are a software engineering assistant specialized in bug localization within a code repository. You are provided with:
1. A bug/issue description, which details the problem or error observed by users or developers.
2. A repository file structure, which includes filenames and directory paths.

Your task is to identify which file(s) in the project are most likely to contain the defect described by the given issue.
1. Carefully read the text describing the bug or problem.
2. Identify any explicit references to files, classes, functions, or specific behaviors that might indicate where the bug resides.
3. Review the list of all files and directories in the project.
4. Consider the naming conventions, the likely contents or responsibilities of each file (e.g., front-end vs. back-end, utility libraries, etc.), and any file that may be relevant given the issue description.
5. Based on the clues in the issue description, hypothesize which files or modules are most likely responsible for the observed bug.
6. If no explicit file is mentioned, rely on logical inferences (e.g., a UI bug might be in front-end components, a database connectivity issue might be in the database configuration or service layer, etc.).

Output Format
- You must return exactly {config['top_k']} file paths, ordered from the highest chance of containing the bug to the lowest.
- Your response should include ONLY files full paths relative to the root of the repository. 
- Do not include explanations or any other text outside of this strict format.
- Use the exact format below (replace [file path] with the actual path or filename):
```
<top_1_path>
[file path for rank #1]
</top_1_path>
<top_2_path>
[file path for rank #2]
</top_2_path>
...
```
""", 
        
        "claude": f"""
You are a specialized code analysis assistant tasked with identifying potentially buggy files in a software project based on GitHub issue descriptions. Your primary goal is LOCALIZATION - determining which specific files in the codebase likely contain the bug described in the issue.

## INPUT CONTEXT
You will be provided with:
1. A GitHub issue description detailing a bug or problem
2. The project's file structure showing available files and directories

## EXPECTED OUTPUT
- Provide a ranked list of files that are most likely to contain the bug. 
- List the top {config['top_k']} most likely locations of the bug in order of probability.
- Your response should include ONLY files full paths relative to the root of the repository. 
- Do not include explanations or any other text outside of this strict format.
- Your output should follow this format:
    ```
    <top_1_path>
    [file path for rank #1]
    </top_1_path>
    <top_2_path>
    [file path for rank #2]
    </top_2_path>
    ...
    ```

## LOCALIZATION STRATEGIES
Use these strategies to identify relevant files:
1. Error Tracing: Analyze error messages, stack traces, or logs to determine the execution path during failure
2. Keyword Matching: Identify domain-specific terms in the issue that might correspond to specific components
3. Function/Feature Mapping: Map the described functionality to likely implementation files
4. File Naming Conventions: Use file naming patterns to identify relevant modules
5. Import/Dependency Analysis: Consider which files might be imported by core functionality
6. Control Flow Understanding: Consider the execution flow described in the issue


## REASONING APPROACH
1. First, thoroughly understand the issue description
2. Identify key symptoms, errors, or unexpected behaviors
3. Map these to potential areas of code based on the file structure
4. Consider both direct causes and potential interaction effects
5. Rank files based on likelihood of containing the bug

## CONSTRAINTS AND GUIDELINES
1. Consider the project's architecture and typical patterns for similar codebases
2. Limit your analysis to the files visible in the provided file structure

""", 

        "mistral": f"""
You are a specialized AI model designed to assist in identifying potential buggy files within a software project based on an issue description and the project's file structure. Your task is to analyze the provided issue description and the structure of the repository to determine the most likely files that contain the bug.

Instructions:
1. Understand the Issue Description: Carefully read the issue description to grasp the nature of the problem. Pay attention to keywords, error messages, and any specific functionalities or components mentioned.
2. Analyze the File Structure: Examine the structure of the repository. Identify files that are relevant to the issue based on their names, paths, and extensions. Consider the typical organization of the project and the roles of different files.
3. Identify Potential Buggy Files: Based on your analysis, identify the top 10 files that are most likely to contain the bug. Prioritize files that are directly related to the issue description and those that are commonly associated with the mentioned functionalities or components.
4. Output the Results: Provide the top {config['top_k']} file paths in the following format, without any additional explanation or text:
```
<top_1_path>
[file path for rank #1]
</top_1_path>
<top_2_path>
[file path for rank #2]
</top_2_path>
...
<top_{config['top_k']}_path>
[file path for rank #{config['top_k']}]
</top_{config['top_k']}_path>
```

""", 
        "deepseek": "", 
        "grok": f"""
You are an AI assistant tasked with identifying potential buggy files in a software project based on a given issue description and the project's file structure. This task is the localization step in solving software engineering problems, specifically for the SWE-Bench dataset, which evaluates LLMs on real-world GitHub issues.

## Input
You will receive two pieces of information:
1. Issue Description: A text describing the problem or bug in the software.
2. Project's File Structure: A list of file paths present in the project, representing the files and directories in the software repository.

## Task
Your goal is to analyze the issue description and determine which files in the project are most likely to contain the bug. You should:
- Output a ranked list of file paths, with the most likely file first.
- Select file paths only from the provided list in the project's file structure.
- Provide {config['top_k']} file paths as you deem relevant.

## Output Format
Your response must consist solely of the ranked list of file paths in the following format, with no additional explanations or commentary:
```
<top_1_path>
[file path for rank #1]
</top_1_path>
<top_2_path>
[file path for rank #2]
</top_2_path>
...
```

"""
    }

    if config["simple_prompt"]:
        prompts[config["llm"]] = ""
    if config["improved_simple_prompt"]:
        prompts[config["llm"]] = ""

    return prompts[config["llm"]]

def create_config(updated_config):
    """Create configuration dictionary for execution parameters"""
    config = {
        # "llm": updated_config["llm"],        # Options: "chatgpt", "claude", "deepseek", "gemini", "mistral", "grok"
        "top_k": 10,            # Number of candidate buggy files to return
        "show_descriptions": False,  # Whether to include module descriptions in tree
        "ignored_dirs": ["config", "test", "tests", "example", "examples"],
        "file_extensions": ['.py'],
        "temperature": 0,       # Lower temperature for more deterministic outputs
        "hierarchy": 0,
        "max_tokens": {         # Maximum tokens for each model
            "chatgpt": 4096,
            "claude": 4096,
            "deepseek": 4096,
            "gemini": 4096,
            "mistral": 4096,
            "grok": 4096
        },
        "use_json_tree": False # Whether to use JSON tree format instead of string
    }

    for key in updated_config.keys():
        config[key] = updated_config[key]
    
    config["system_prompt"] = get_system_prompt(config)

    return config

In [ ]:
def create_llms(config):
    """Create language model instances based on configuration"""
    llms = {

    }
    llm_name = config["llm"]

    if llm_name == "chatgpt":
        llms[llm_name] = ChatOpenAI(
            # model_name="gpt-4o",
            model_name=config["model_name"],
            openai_api_key=os.getenv("OPENAI_API_KEY"),
            temperature=config["temperature"],
            max_tokens=config["max_tokens"]["chatgpt"]
        )
    elif llm_name == "claude":
        llms[llm_name] = ChatAnthropic(
            # model_name="claude-3-5-sonnet-20240620",
            model_name=config["model_name"],
            anthropic_api_key=os.getenv("ANTHROPIC_API_KEY"),
            temperature=config["temperature"],
            max_tokens=config["max_tokens"]["claude"]
        )
    elif llm_name == "deepseek":
        llms[llm_name] = ChatDeepSeek(
            # model="deepseek-chat",
            model=config["model_name"],
            api_key=os.getenv("DEEPSEEK_API_KEY"),
            temperature=config["temperature"],
            max_tokens=config["max_tokens"]["deepseek"]
        )
    elif llm_name == "gemini":
        llms[llm_name] = ChatGoogleGenerativeAI(
            # model="gemini-1.5-pro",
            model=config["model_name"],
            google_api_key=os.getenv("GOOGLE_API_KEY"),
            temperature=config["temperature"],
            max_tokens=config["max_tokens"]["gemini"]
        )
    elif llm_name == "mistral":
        llms[llm_name] = ChatMistralAI(
            # model="mistral-large-latest",
            model=config["model_name"],
            mistral_api_key=os.getenv("MISTRAL_API_KEY"),
            temperature=config["temperature"],
            max_tokens=config["max_tokens"]["mistral"]
        )
    elif llm_name == "grok":
        llms[llm_name] = ChatXAI(
            # model="grok-2-1212",
            model=config["model_name"],
            api_key=os.getenv("XAI_API_KEY"),
            temperature=config["temperature"],
            max_tokens=config["max_tokens"]["grok"]
        )
    elif llm_name == "nvidia":
        llms[llm_name] = ChatNVIDIA(
            model=config["model_name"],
            api_key=os.getenv("NVIDIA_API_KEY"),
            temperature=config["temperature"],
            max_tokens=config["max_tokens"]["grok"]
        )

    else:
        raise ValueError(f"Unsupported LLM: {llm_name}")

    return llms

In [ ]:
def initialize_output_directories(config):
    """Create output directories based on configuration"""
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    base_path = f"./localization/hierarchy_{config['hierarchy']}/{config['llm']}/{timestamp}"
    
    # Create directories
    os.makedirs(f"{base_path}/raw_outputs", exist_ok=True)
    
    # Save config
    with open(f"{base_path}/config.json", 'w') as f:
        json.dump(config, f, indent=4)
        
    return base_path

In [ ]:
@retry(
    stop=stop_after_attempt(10),
    wait=wait_exponential(multiplier=1, min=30, max=300),
    retry=retry_if_exception_type((Exception))
)
def query_llm(prompt, llm_name, config, llms):
    """Query the specified LLM with retry logic."""
    try:
        llm = llms[llm_name]
        
        # Build messages depending on whether the model supports system messages
        messages = []
        
        # Models that support system messages
        if llm_name in ["chatgpt", "claude", "gemini", "mistral", "grok"]:
            if config["system_prompt"]:
                messages.append(SystemMessage(content=config["system_prompt"]))
            messages.append(HumanMessage(content=prompt))
        else:
            # For models without system message support (deepseek, grok)
            messages.append(HumanMessage(content=prompt))
        
        response = llm.invoke(messages)
        return response.content
    except Exception as e:
        print(f"Error querying {llm_name}: {str(e)}")
        raise

In [ ]:
def get_user_prompt(problem_statement, tree_section, config):
        
    simple_prompt = f"""
# Bug Localization Task

You are an expert software engineer tasked with identifying potential buggy files in a Python codebase.

## Problem Statement
{problem_statement}

## Codebase Structure
Below is the directory structure of the codebase:
```
{tree_section}
```

## Task
Based on the problem statement and codebase structure, identify the top {config['top_k']} files that are most likely to contain the bug. 

Please format your response as follows:

```
<top_1_path>
[file path for rank #1]
</top_1_path>
<top_2_path>
[file path for rank #2]
</top_2_path>
...
```

Your response should include ONLY full paths relative to the root of the codebase.

"""

    improved_simple_prompt = f"""
You are an expert software engineer tasked with identifying potential buggy files in a Python codebase. Your goal is to analyze the given codebase structure and problem statement to pinpoint the most likely locations of a bug.

First, examine the codebase structure provided below:

<codebase_structure>
{tree_section}
</codebase_structure>



Now, carefully read the problem statement describing the bug:

<problem_statement>
{problem_statement}
</problem_statement>



Your task is to identify the top {config['top_k']} files that are most likely to contain the bug based on the problem statement and codebase structure.

After your analysis, provide your final output using the following format:

<top_1_path>
[full file path for rank #1]
</top_1_path>
<top_2_path>
[full file path for rank #2]
</top_2_path>
...

Continue this pattern for all {config['top_k']} files.

Important reminders:
- Include ONLY full paths relative to the root of the codebase.
- Base your decisions solely on the information provided in the problem statement and codebase structure.

"""

    user_prompts = {
        "chatgpt": f"""

Issue Description:
{problem_statement}

Project File Structure:
{tree_section}

List the top {config['top_k']} file paths that are most likely to contain the bug, strictly following the format below:
```
<top_1_path>
[file path for rank #1]
</top_1_path>
<top_2_path>
[file path for rank #2]
</top_2_path>
...
```

""", 
        "deepseek": f"""
# Bug Localization Task

You are an expert software engineer tasked with identifying potential buggy files in a Python codebase.

## Problem Statement
{problem_statement}

## Codebase Structure
Below is the directory structure of the codebase:
```
{tree_section}
```

## Task
Based on the problem statement and codebase structure, identify the top {config['top_k']} files that are most likely to contain the bug. 

Please format your response as follows:

```
<top_1_path>
[file path for rank #1]
</top_1_path>
<top_2_path>
[file path for rank #2]
</top_2_path>
...
```

Your response should include ONLY Python files (.py) with their full paths relative to the root of the codebase.


""",
        "claude": f"""
Given the GitHub issue description and project file structure below, identify the most likely files containing bugs related to this issue.

# Task
Determine which specific files in the repository are most likely to contain the bug described in the issue.

# Bug Description
{problem_statement}

# File Structure
{tree_section}

Based on the bug description and file structure, respond with exactly the top {config["top_k"]} most likely buggy files in order of probability.

Your response must follow this exact format with no additional text or explanations:
```
<top_1_path>
[file path for rank #1]
</top_1_path>
<top_2_path>
[file path for rank #2]
</top_2_path>
...
```

Focus on:
- Files explicitly mentioned in the issue
- Files handling relevant functionality
- Meaningful file naming that matches the bug context
- Code that would logically implement features mentioned
- Files that would contain error-prone logic
""", 

        "mistral": f"""

Issue Description: 
{problem_statement}


Repository File Structure: 
{tree_section}

""", 

        "grok": f"""
Analyze the following issue description and project's file structure to identify potential buggy files:

Issue Description:
{problem_statement}

Project's File Structure:
{tree_section}

"""
    }


    if config["simple_prompt"]:
        user_prompts[config["llm"]] = simple_prompt
    if config["improved_simple_prompt"]:
        user_prompts[config["llm"]] = improved_simple_prompt
    
    
    return user_prompts[config["llm"]]

In [ ]:
def extract_file_paths_from_response(response, instance_id, top_k=10):
    """Extract file paths from LLM response with the new XML tag format."""
    file_paths = []
    
    # New pattern to extract paths from XML tags
    for i in range(1, top_k + 1):
        pattern = rf'<top_{i}_path>(.*?)</top_{i}_path>'
        match = re.search(pattern, response, re.DOTALL)
        if match:
            path = match.group(1).replace(instance_id, '').strip().lstrip('[').rstrip(']').strip('/').strip('\\')
            if path.endswith('.py') and path not in file_paths:
                file_paths.append(path)
    
    # If not enough paths found with tags, fallback to looking for Python files
    if len(file_paths) < top_k:
        path_pattern = r'(?:(?:\d+\.|-|\*|\`)\s*)?([^\s\(\)`]+?\.py)\b'
        matches = re.findall(path_pattern, response)
        
        for match in matches:
            if match.endswith('.py') and match not in file_paths:
                file_paths.append(match)
    
    # Limit to top_k paths
    return file_paths[:top_k]

def create_buggy_file_localization_prompt(problem_statement, tree, config, codebase_path):
    """Create prompt for asking LLM to localize buggy files."""
    
    # Decide which tree format to use
    if config["use_json_tree"]:
        tree_section = f"```json\n{tree}\n```"
    else:
        tree_section = f"```\n{tree}\n```"
    
    # print(tree_section)
    # print()
    
    return get_user_prompt(problem_statement, tree_section, config)

In [ ]:
def save_raw_response(response, instance_id, output_path):
    """Save raw LLM response to a file."""
    with open(f"{output_path}/raw_outputs/{instance_id}.txt", 'w', encoding='utf-8') as f:
        f.write(response)

def aggregate_predictions(output_path, config):
    """Aggregate predictions from saved raw responses and write to predictions.json."""
    predictions = {}
    raw_outputs_dir = os.path.join(output_path, "raw_outputs")
    for filename in os.listdir(raw_outputs_dir):
        if filename.endswith(".txt"):
            instance_id = filename[:-4]  # remove '.txt'
            with open(os.path.join(raw_outputs_dir, filename), 'r', encoding='utf-8') as f:
                response = f.read()
            predicted_file_paths = extract_file_paths_from_response(response, instance_id, config["top_k"])
            predictions[instance_id] = predicted_file_paths
    predictions_file = os.path.join(output_path, "predictions.json")
    with open(predictions_file, 'w') as f:
        json.dump(predictions, f, indent=4)

def evaluate_predictions(predictions_file, ground_truth_file, top_k_values=None):
    """Evaluate prediction accuracy for different top-k values."""
    if top_k_values is None:
        top_k_values = [1, 3, 5, 7, 10]
    
    # Load predictions and ground truth
    with open(predictions_file, 'r') as f:
        predictions = json.load(f)
    
    with open(ground_truth_file, 'r') as f:
        ground_truth = json.load(f)
    
    # Calculate accuracy for each top-k value
    results = {}
    for k in top_k_values:
        correct = 0
        total = 0
        
        for instance_id, true_path in ground_truth.items():
            if instance_id in predictions:
                total += 1
                
                # Check if ground truth is in top-k predictions
                predicted_paths = predictions[instance_id][:k]
                if any(true_path.endswith(path) or path.endswith(true_path) for path in predicted_paths):
                    correct += 1
        
        if total > 0:
            accuracy = correct / total
            results[f"top_{k}"] = {
                "accuracy": accuracy,
                "correct": correct,
                "total": total
            }
    
    return results

In [ ]:
def process_instance(task, config, output_path, extensions_dict=None):
    """Process a single dataset instance in a separate process."""
    instance_id = task["instance_id"]
    problem_statement = task["problem_statement"]
    patch = task["patch"]
    
    # print(f"Processing {instance_id}...")
    
    # Extract ground truth file path from patch
    ground_truth_file_path = extract_modified_file_path(patch)
    if not ground_truth_file_path:
        print(f"[Warning] No file path found for {instance_id}")
        return None
    
    # Check if ground truth file exists
    codebase_path = f"./codebases/{instance_id}"
    file_full_path = f"{codebase_path}/{ground_truth_file_path}"
    if not os.path.exists(file_full_path):
        print(f"[Error] Ground truth file not found: {file_full_path}")
        return None
    
    # Get file extensions for this instance_id from the extensions dictionary
    file_extensions = []
    if extensions_dict and instance_id in extensions_dict:
        file_extensions = extensions_dict[instance_id]
    elif extensions_dict:
        print(f"[Warning] No extensions found for {instance_id} in extensions JSON")
    
    
    if ".txt" in file_extensions:
        file_extensions.remove(".txt")
    if ".md" in file_extensions:
        file_extensions.remove(".md")
    if ".html" in file_extensions:
        file_extensions.remove(".html")

    # Generate tree representation
    if config["use_json_tree"]:
        tree = generate_tree_json(
            codebase_path,
            show_descriptions=config["show_descriptions"],
            file_extensions=file_extensions,  # Use instance-specific extensions
            ignored_dirs=config["ignored_dirs"]
        )
    else:
        tree = generate_tree_string(
            codebase_path,
            show_descriptions=config["show_descriptions"],
            file_extensions=file_extensions,  # Use instance-specific extensions
            ignored_dirs=config["ignored_dirs"]
        )
    
    # Create prompt for the LLM
    prompt = create_buggy_file_localization_prompt(problem_statement, tree, config, codebase_path)
    
    # Create LLM instances inside the process so nothing unpickleable is passed in
    local_llms = create_llms(config)
    response = query_llm(prompt, config["llm"], config, local_llms)

    try:
        # Save raw response
        save_raw_response(response, instance_id, output_path)
        
        # Extract file paths from the response
        predicted_file_paths = extract_file_paths_from_response(response, instance_id, config["top_k"])
        
        # Check if ground truth is in the predicted file paths
        gt_path = Path(ground_truth_file_path).resolve()
        predicted_paths = [Path(p).resolve() for p in predicted_file_paths]

        is_correct = gt_path in predicted_paths
        
        result = {
            "instance_id": instance_id,
            "ground_truth": ground_truth_file_path,
            "predictions": predicted_file_paths,
            "is_correct": is_correct
        }
            
        return result
        
    except Exception as e:
        print(f"[Error] Exception occurred for {instance_id}: {str(e)}")
        return None

In [ ]:
def localize_buggy_files(config_params):
    """Main function to localize buggy files in the SWE-Bench dataset."""
    # Load environment
    load_environment()
    
    # Create config
    config = create_config(config_params)
    
    # Update config with any provided parameters
    for key, value in config_params.items():
        if key in config:
            config[key] = value
    
    # Initialize output directories
    output_path = initialize_output_directories(config)
    
    # Load the SWE-bench_Lite dataset
    dataset = load_dataset("princeton-nlp/SWE-bench_Lite", split="test")
    
    # Load file extensions JSON once at the beginning
    extensions_dict = {}
    if config["file_extensions_json"]:
        try:
            with open(config["file_extensions_json"], 'r') as f:
                extensions_dict = json.load(f)
                print(f"Loaded file extensions for {len(extensions_dict)} instances")
        except Exception as e:
            print(f"[Error] Failed to load extensions from JSON: {str(e)}")
    
    # Set up multiprocessing
    num_processes = config["num_processes"]
    print(f"Using {num_processes} processes for parallel processing")
    
    # Use partial without passing non-pickleable objects like llms
    with multiprocessing.Pool(processes=num_processes) as pool:
        process_func = partial(process_instance, config=config, output_path=output_path, extensions_dict=extensions_dict)
        results = list(tqdm(pool.imap(process_func, dataset), total=len(dataset), desc="Processing instances"))
    
    # Filter out None results (failed instances)
    results = [r for r in results if r is not None]
    
    # Aggregate predictions from raw outputs into predictions.json
    aggregate_predictions(output_path, config)
    
    # Calculate overall metrics
    total_instances = len(results)
    correct_localizations = sum(1 for r in results if r["is_correct"])
    
    if total_instances > 0:
        accuracy = correct_localizations / total_instances
        print(f"\nSummary:")
        print(f"Total instances processed: {total_instances}")
        print(f"Correct localizations: {correct_localizations}")
        print(f"Accuracy: {accuracy:.2f} ({correct_localizations}/{total_instances})")
    
    evaluation_results = evaluate_predictions(
        f"{output_path}/predictions.json", 
        "./ground_truth/bug_paths.json",
        [1, 3, 5, 7, 10, 12, 15]
    )
    
    with open(f"{output_path}/evaluation_results.json", 'w') as f:
        json.dump(evaluation_results, f, indent=4)
    
    print("\nEvaluation Results:")
    for k, result in evaluation_results.items():
        print(f"{k}: {result['accuracy']:.2f} ({result['correct']}/{result['total']})")
    
    return evaluation_results

In [ ]:
config_params = {
    "llm": "deepseek",        # Options: "chatgpt", "claude", "deepseek", "gemini", "mistral", "grok"
    "model_name": "deepseek-chat", 
    "hierarchy": 777,
    "top_k": 15,
    "temperature": 0,
    "num_processes": 16, 
    "show_descriptions": False, 
    "use_json_tree": False, 
    "simple_prompt": True, 
    "improved_simple_prompt": False, 
    "ignored_dirs": [],
    "file_extensions_json": "./file_extensions/20250402_164857/results.json"
}
localize_buggy_files(config_params)

In [ ]:
# max(1, multiprocessing.cpu_count())

In [ ]:
# evaluation_results = evaluate_predictions(
#     f"{output_path}/predictions.json", 
#     "/home/tweichuan/project/ground_truth/bug_paths.json",
#     [1, 3, 5, 7, 10]
# )

In [ ]:
# Deekseep False False
# {'top_1': {'accuracy': 0.6566666666666666, 'correct': 197, 'total': 300},
#  'top_3': {'accuracy': 0.84, 'correct': 252, 'total': 300},
#  'top_5': {'accuracy': 0.8733333333333333, 'correct': 262, 'total': 300},
#  'top_7': {'accuracy': 0.9166666666666666, 'correct': 275, 'total': 300},
#  'top_10': {'accuracy': 0.9433333333333334, 'correct': 283, 'total': 300}}

# GPT 4 mini True True
# {'top_1': {'accuracy': 0.47, 'correct': 141, 'total': 300},
#  'top_3': {'accuracy': 0.62, 'correct': 186, 'total': 300},
#  'top_5': {'accuracy': 0.66, 'correct': 198, 'total': 300},
#  'top_7': {'accuracy': 0.69, 'correct': 207, 'total': 300},
#  'top_10': {'accuracy': 0.7366666666666667, 'correct': 221, 'total': 300}}

# GPT 4 mini False Fales
# {'top_1': {'accuracy': 0.49666666666666665, 'correct': 149, 'total': 300},
#  'top_3': {'accuracy': 0.64, 'correct': 192, 'total': 300},
#  'top_5': {'accuracy': 0.6966666666666667, 'correct': 209, 'total': 300},
#  'top_7': {'accuracy': 0.73, 'correct': 219, 'total': 300},
#  'top_10': {'accuracy': 0.76, 'correct': 228, 'total': 300}}

# GPT 4 mini False False, no system_prompt
# {'top_1': {'accuracy': 0.47333333333333333, 'correct': 142, 'total': 300},
#  'top_3': {'accuracy': 0.6566666666666666, 'correct': 197, 'total': 300},
#  'top_5': {'accuracy': 0.7233333333333334, 'correct': 217, 'total': 300},
#  'top_7': {'accuracy': 0.75, 'correct': 225, 'total': 300},
#  'top_10': {'accuracy': 0.76, 'correct': 228, 'total': 300}}

# GPT 4 mini False False, no system_prompt, simple user prompt
# {'top_1': {'accuracy': 0.43666666666666665, 'correct': 131, 'total': 300},
#  'top_3': {'accuracy': 0.6133333333333333, 'correct': 184, 'total': 300},
#  'top_5': {'accuracy': 0.6966666666666667, 'correct': 209, 'total': 300},
#  'top_7': {'accuracy': 0.7266666666666667, 'correct': 218, 'total': 300},
#  'top_10': {'accuracy': 0.76, 'correct': 228, 'total': 300}}

# claude-3-haiku-20240307 False False
# {'top_1': {'accuracy': 0.4866666666666667, 'correct': 146, 'total': 300},
#  'top_3': {'accuracy': 0.61, 'correct': 183, 'total': 300},
#  'top_5': {'accuracy': 0.6966666666666667, 'correct': 209, 'total': 300},
#  'top_7': {'accuracy': 0.7266666666666667, 'correct': 218, 'total': 300},
#  'top_10': {'accuracy': 0.76, 'correct': 228, 'total': 300}}

# mistral-large-latest False False
# {'top_1': {'accuracy': 0.5052264808362369, 'correct': 145, 'total': 287},
#  'top_3': {'accuracy': 0.686411149825784, 'correct': 197, 'total': 287},
#  'top_5': {'accuracy': 0.7491289198606271, 'correct': 215, 'total': 287},
#  'top_7': {'accuracy': 0.7700348432055749, 'correct': 221, 'total': 287},
#  'top_10': {'accuracy': 0.7979094076655052, 'correct': 229, 'total': 287}}

# mistral-large-latest True False
# {'top_1': {'accuracy': 0.4666666666666667, 'correct': 140, 'total': 300},
#  'top_3': {'accuracy': 0.6666666666666666, 'correct': 200, 'total': 300},
#  'top_5': {'accuracy': 0.74, 'correct': 222, 'total': 300},
#  'top_7': {'accuracy': 0.7566666666666667, 'correct': 227, 'total': 300},
#  'top_10': {'accuracy': 0.7966666666666666, 'correct': 239, 'total': 300}}

# mistral-large-latest False False simple_prompt
# {'top_1': {'accuracy': 0.48333333333333334, 'correct': 145, 'total': 300},
#  'top_3': {'accuracy': 0.68, 'correct': 204, 'total': 300},
#  'top_5': {'accuracy': 0.7133333333333334, 'correct': 214, 'total': 300},
#  'top_7': {'accuracy': 0.7566666666666667, 'correct': 227, 'total': 300},
#  'top_10': {'accuracy': 0.8133333333333334, 'correct': 244, 'total': 300}}

# mistral codestral-latest False False
# {'top_1': {'accuracy': 0.5033333333333333, 'correct': 151, 'total': 300},
#  'top_3': {'accuracy': 0.6866666666666666, 'correct': 206, 'total': 300},
#  'top_5': {'accuracy': 0.7433333333333333, 'correct': 223, 'total': 300},
#  'top_7': {'accuracy': 0.77, 'correct': 231, 'total': 300},
#  'top_10': {'accuracy': 0.7933333333333333, 'correct': 238, 'total': 300}}

# grok-2-1212 False False simple_prompt
# {'top_1': {'accuracy': 0.5666666666666667, 'correct': 170, 'total': 300},
#  'top_3': {'accuracy': 0.7566666666666667, 'correct': 227, 'total': 300},
#  'top_5': {'accuracy': 0.8033333333333333, 'correct': 241, 'total': 300},
#  'top_7': {'accuracy': 0.8566666666666667, 'correct': 257, 'total': 300},
#  'top_10': {'accuracy': 0.8766666666666667, 'correct': 263, 'total': 300}}

# grok-2-1212 False False system prompt
# {'top_1': {'accuracy': 0.55, 'correct': 165, 'total': 300},
#  'top_3': {'accuracy': 0.73, 'correct': 219, 'total': 300},
#  'top_5': {'accuracy': 0.7733333333333333, 'correct': 232, 'total': 300},
#  'top_7': {'accuracy': 0.8, 'correct': 240, 'total': 300},
#  'top_10': {'accuracy': 0.84, 'correct': 252, 'total': 300}}

# claude-3-haiku-20240307 False False
# {'top_1': {'accuracy': 0.48333333333333334, 'correct': 145, 'total': 300},
#  'top_3': {'accuracy': 0.6333333333333333, 'correct': 190, 'total': 300},
#  'top_5': {'accuracy': 0.6933333333333334, 'correct': 208, 'total': 300},
#  'top_7': {'accuracy': 0.73, 'correct': 219, 'total': 300},
#  'top_10': {'accuracy': 0.7833333333333333, 'correct': 235, 'total': 300}}

# claude-3-haiku-20240307 True False
# {'top_1': {'accuracy': 0.43, 'correct': 129, 'total': 300},
#  'top_3': {'accuracy': 0.5866666666666667, 'correct': 176, 'total': 300},
#  'top_5': {'accuracy': 0.65, 'correct': 195, 'total': 300},
#  'top_7': {'accuracy': 0.7233333333333334, 'correct': 217, 'total': 300},
#  'top_10': {'accuracy': 0.7533333333333333, 'correct': 226, 'total': 300}}

# claude-3-haiku-20240307 False True
# {'top_1': {'accuracy': 0.5066666666666667, 'correct': 152, 'total': 300},
#  'top_3': {'accuracy': 0.6566666666666666, 'correct': 197, 'total': 300},
#  'top_5': {'accuracy': 0.7233333333333334, 'correct': 217, 'total': 300},
#  'top_7': {'accuracy': 0.7466666666666667, 'correct': 224, 'total': 300},
#  'top_10': {'accuracy': 0.7866666666666666, 'correct': 236, 'total': 300}}

# claude-3-haiku-20240307 True True
# {'top_1': {'accuracy': 0.49, 'correct': 147, 'total': 300},
#  'top_3': {'accuracy': 0.6666666666666666, 'correct': 200, 'total': 300},
#  'top_5': {'accuracy': 0.7033333333333334, 'correct': 211, 'total': 300},
#  'top_7': {'accuracy': 0.7633333333333333, 'correct': 229, 'total': 300},
#  'top_10': {'accuracy': 0.7833333333333333, 'correct': 235, 'total': 300}}

# claude-3-haiku-20240307 False False simple_prompt
# {'top_1': {'accuracy': 0.49333333333333335, 'correct': 148, 'total': 300},
#  'top_3': {'accuracy': 0.6666666666666666, 'correct': 200, 'total': 300},
#  'top_5': {'accuracy': 0.7333333333333333, 'correct': 220, 'total': 300},
#  'top_7': {'accuracy': 0.7566666666666667, 'correct': 227, 'total': 300},
#  'top_10': {'accuracy': 0.8133333333333334, 'correct': 244, 'total': 300}}

# gpt-4o-2024-08-06 False False
# {'top_1': {'accuracy': 0.58, 'correct': 174, 'total': 300},
#  'top_3': {'accuracy': 0.75, 'correct': 225, 'total': 300},
#  'top_5': {'accuracy': 0.8133333333333334, 'correct': 244, 'total': 300},
#  'top_7': {'accuracy': 0.8433333333333334, 'correct': 253, 'total': 300},
#  'top_10': {'accuracy': 0.8833333333333333, 'correct': 265, 'total': 300}}

# gpt-4o-2024-08-06 False False simple_prompt
# {'top_1': {'accuracy': 0.6, 'correct': 180, 'total': 300},
#  'top_3': {'accuracy': 0.7666666666666667, 'correct': 230, 'total': 300},
#  'top_5': {'accuracy': 0.8066666666666666, 'correct': 242, 'total': 300},
#  'top_7': {'accuracy': 0.8533333333333334, 'correct': 256, 'total': 300},
#  'top_10': {'accuracy': 0.8866666666666667, 'correct': 266, 'total': 300}}

# gpt-4o-2024-08-06 True False simple_prompt
# {'top_1': {'accuracy': 0.61, 'correct': 183, 'total': 300},
#  'top_3': {'accuracy': 0.7633333333333333, 'correct': 229, 'total': 300},
#  'top_5': {'accuracy': 0.8233333333333334, 'correct': 247, 'total': 300},
#  'top_7': {'accuracy': 0.8833333333333333, 'correct': 265, 'total': 300},
#  'top_10': {'accuracy': 0.9033333333333333, 'correct': 271, 'total': 300}}

# gpt-4o-2024-08-06 True True simple_prompt
# {'top_1': {'accuracy': 0.5733333333333334, 'correct': 172, 'total': 300},
#  'top_3': {'accuracy': 0.7866666666666666, 'correct': 236, 'total': 300},
#  'top_5': {'accuracy': 0.84, 'correct': 252, 'total': 300},
#  'top_7': {'accuracy': 0.88, 'correct': 264, 'total': 300},
#  'top_10': {'accuracy': 0.9033333333333333, 'correct': 271, 'total': 300}}

# claude-3-5-sonnet-20241022 False False
# {'top_1': {'accuracy': 0.6133333333333333, 'correct': 184, 'total': 300},
#  'top_3': {'accuracy': 0.8066666666666666, 'correct': 242, 'total': 300},
#  'top_5': {'accuracy': 0.87, 'correct': 261, 'total': 300},
#  'top_7': {'accuracy': 0.9, 'correct': 270, 'total': 300},
#  'top_10': {'accuracy': 0.9233333333333333, 'correct': 277, 'total': 300}}

# claude-3-5-sonnet-20241022 False False simple_prompt
# {'top_1': {'accuracy': 0.6533333333333333, 'correct': 196, 'total': 300},
#  'top_3': {'accuracy': 0.82, 'correct': 246, 'total': 300},
#  'top_5': {'accuracy': 0.8766666666666667, 'correct': 263, 'total': 300},
#  'top_7': {'accuracy': 0.91, 'correct': 273, 'total': 300},
#  'top_10': {'accuracy': 0.9366666666666666, 'correct': 281, 'total': 300}}

# claude-3-7-sonnet-20250219 False False simle_prompt
# {'top_1': {'accuracy': 0.69, 'correct': 207, 'total': 300},
#  'top_3': {'accuracy': 0.8466666666666667, 'correct': 254, 'total': 300},
#  'top_5': {'accuracy': 0.8966666666666666, 'correct': 269, 'total': 300},
#  'top_7': {'accuracy': 0.92, 'correct': 276, 'total': 300},
#  'top_10': {'accuracy': 0.95, 'correct': 285, 'total': 300}}